## DATA INGESTION

In [1]:
import os
from pathlib import Path
import magic
import fitz
import pandas as pd
from tqdm import tqdm

In [2]:
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)


In [3]:
def is_pdf(file_path):
    try:
        file_type = magic.from_file(str(file_path), mime=True)
        print(f"Detected MIME for {file_path}: {file_type}")
        return file_type == "application/pdf"
    except Exception as e:
        print("Error:",e)
        return False

In [4]:
is_pdf("../data/raw/sample.pdf")

Detected MIME for ../data/raw/sample.pdf: application/pdf


True

In [5]:
def extract_pdf_metadata(pdf_path):
    pdf_path = Path(pdf_path)
    try:
        doc = fitz.open(pdf_path)
        meta = doc.metadata
        info = {
            "file_name": pdf_path.name,
            "path": str(pdf_path),
            "pages": len(doc),
            "title": meta.get("title", None),
            "author": meta.get("author", None),
            "filesize_kb": round(os.path.getsize(pdf_path) / 1024, 2),
        }
        doc.close()
        return info
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return None

In [6]:
result = extract_pdf_metadata("../data/raw/sample.pdf")
result

{'file_name': 'sample.pdf',
 'path': '../data/raw/sample.pdf',
 'pages': 19,
 'title': 'AutoFactory Dataset to Support AI in Manufacturing Systems',
 'author': 'Abderrahmane Boudribila',
 'filesize_kb': 2327.14}

In [7]:
pdf_files = list(raw_data_dir.glob("*.pdf"))
pdf_files

[PosixPath('../data/raw/sample.pdf')]

In [8]:
metadata_list = []

for pdf in tqdm(pdf_files):
    meta = extract_pdf_metadata(pdf)
    if meta:
        metadata_list.append(meta)

df_metadata = pd.DataFrame(metadata_list)
df_metadata
        
    


100%|██████████| 1/1 [00:00<00:00, 762.74it/s]


,file_name,path,pages,title,author,filesize_kb
0,sample.pdf,../data/raw/sample.pdf,19,AutoFactory Dataset to Support AI in Manufactu...,Abderrahmane Boudribila,2327.14


In [9]:
df_metadata.to_csv("../data/raw/metadata.csv", index=False)
print("saved metadata.csv")

saved metadata.csv
